In [ ]:
import sys; sys.path.insert(0, "..")
from fantasy.yahoo_client import get_my_roster, get_free_agents, get_current_matchup, get_league_categories
from fantasy.nba_stats import get_player_stats, get_games_this_week
from fantasy.analysis import project_team_categories, project_player
from fantasy.llm import build_waiver_prompt, ask_gemini
import pandas as pd

In [ ]:
matchup = get_current_matchup()
week_start, week_end = matchup["week_start"], matchup["week_end"]
my_roster = get_my_roster()
categories = get_league_categories()  # actual scoring cats like FG%, FT%, 3PTM

my_players = []
for p in my_roster:
    stats = get_player_stats(p["name"])
    games = get_games_this_week(p["team_abbr"], week_start, week_end)
    my_players.append({**p, "stats": stats, "games_remaining": games})

my_totals = project_team_categories(my_players)

def to_league_cats(proj, cats):
    """Map raw stat projections to league scoring category values."""
    result = {}
    for cat in cats:
        if cat in proj:
            result[cat] = proj[cat]
        elif cat == "FG%":
            result[cat] = proj["FGM"] / proj["FGA"] if proj.get("FGA", 0) > 0 else 0.0
        elif cat == "FT%":
            result[cat] = proj["FTM"] / proj["FTA"] if proj.get("FTA", 0) > 0 else 0.0
        elif cat == "3PTM":
            result[cat] = proj.get("FG3M", 0.0)
    return result

my_cats = to_league_cats(my_totals, categories)

# Typical weekly team projections used to normalize across cat types
TYPICAL_WEEKLY = {
    "PTS": 900, "REB": 400, "AST": 220, "STL": 90, "BLK": 60,
    "TOV": 175, "3PTM": 120, "FG%": 0.46, "FT%": 0.78,
}

def weakness_score(cat, val):
    """Returns a normalized weakness score. Lower = weaker. TOV is inverted."""
    typical = TYPICAL_WEEKLY.get(cat, 1.0)
    ratio = val / typical if typical > 0 else 0.0
    return -ratio if cat == "TOV" else ratio  # TOV: higher val = weaker

sorted_cats = sorted(categories, key=lambda c: weakness_score(c, my_cats.get(c, 0)))
weak_cats = sorted_cats[:3]
print(f"Weakest categories: {weak_cats}")

In [ ]:
fas = get_free_agents(count=50)

scored = []
for fa in fas:
    stats = get_player_stats(fa["name"])
    games = get_games_this_week(fa["team_abbr"], week_start, week_end)
    if stats is None:
        continue
    proj = project_player(stats, games, fa.get("status", ""))
    fa_cats = to_league_cats(proj, categories)
    # Fit score: normalized contribution in weak categories (TOV inverted)
    fit = 0.0
    for c in weak_cats:
        typical = TYPICAL_WEEKLY.get(c, 1.0)
        val = fa_cats.get(c, 0)
        if c == "TOV":
            fit += (typical - val) / typical  # reward low TOV
        else:
            fit += val / typical
    scored.append({**fa, "games_remaining": games, "projected": proj, "fit_score": fit})

scored.sort(key=lambda x: x["fit_score"], reverse=True)
print(pd.DataFrame([{
    "Player": p["name"], "Pos": p["position"], "Status": p["status"] or "Active",
    "Games": p["games_remaining"], "Fit Score": round(p["fit_score"], 2)
} for p in scored[:20]]).to_string(index=False))

In [ ]:
prompt = build_waiver_prompt(scored, weak_cats)
advice = ask_gemini(prompt)
print("\n=== WAIVER WIRE PICKUPS ===\n")
print(advice)